In [ ]:
from pathlib import Path
import subprocess
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')])


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
print(f'Seed fixed at {SEED}')
print(f'Project root: {PROJECT_ROOT}')


# Notebook 04 ? Multi-Hazard Patch Generation
## Exports aligned VV/VH/terrain inputs with hazard target stacks


## Section 4.1 — Multi-Input Patching Function

Patch extraction converts scene-level geospatial products into model-ready tensors. Because CASA-Net uses separate VV, VH, and terrain streams, this step must preserve alignment across all inputs and the mask.


In [ ]:
import json
from pathlib import Path

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

PROCESSED_SAR = PROCESSED_DIR / 'sar'
PROCESSED_TERRAIN = PROCESSED_DIR / 'terrain'
PROCESSED_MASKS = PROCESSED_DIR / 'masks'
PROCESSED_PATCHES = PROCESSED_DIR / 'patches'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORT_DIR = OUTPUTS_DIR / 'report'
PROCESSED_PATCHES.mkdir(parents=True, exist_ok=True)

PATCH_SIZE = 256
OVERLAP = 64
STRIDE = PATCH_SIZE - OVERLAP
terrain_dict = {
    'slope': PROCESSED_TERRAIN / 'slope_sylhet.tif',
    'twi': PROCESSED_TERRAIN / 'twi_sylhet.tif',
    'jrc': PROCESSED_TERRAIN / 'jrc_water_sylhet.tif',
    'hand': PROCESSED_TERRAIN / 'hand_sylhet.tif',
}

def iter_windows(width, height, patch_size=256, stride=192):
    for row_off in range(0, max(height - patch_size + 1, 1), stride):
        for col_off in range(0, max(width - patch_size + 1, 1), stride):
            yield Window(col_off, row_off, patch_size, patch_size)
    yield Window(max(width - patch_size, 0), max(height - patch_size, 0), patch_size, patch_size)

def pad(arr, channels):
    out = np.zeros((channels, PATCH_SIZE, PATCH_SIZE), dtype=arr.dtype)
    out[:, :arr.shape[1], :arr.shape[2]] = arr
    return out

def generate_patches(vv_tif, vh_tif, terrain_dict, mask_tif, output_dir):
    rows = []
    with rasterio.open(vv_tif) as vv_src, rasterio.open(vh_tif) as vh_src, rasterio.open(mask_tif) as mask_src:
        terrain_sources = {name: rasterio.open(path) for name, path in terrain_dict.items()}
        try:
            for idx, window in enumerate(iter_windows(vv_src.width, vv_src.height, PATCH_SIZE, STRIDE)):
                vv = vv_src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32)[None, ...]
                vh = vh_src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32)[None, ...]
                terrain = np.stack([src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32) for src in terrain_sources.values()], axis=0)
                mask_arr = mask_src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32)[None, ...]
                vv = pad(vv, 1)
                vh = pad(vh, 1)
                terrain = pad(terrain, 4)
                mask_arr = pad(mask_arr, 1)
                nodata_ratio = float(np.mean((vv == 0) & (vh == 0)))
                flood_pct = float(mask_arr.mean())
                if nodata_ratio > 0.90 or flood_pct == 0.0:
                    continue
                patch_path = output_dir / f"{Path(vv_tif).stem}_{idx:04d}.npz"
                np.savez_compressed(patch_path, vv=vv.astype(np.float32), vh=vh.astype(np.float32), terrain=terrain.astype(np.float32), mask=mask_arr.astype(np.float32))
                rows.append({'patch_path': str(patch_path), 'source_date': Path(vv_tif).stem.split('_')[-1], 'flood_pixel_pct': flood_pct, 'nodata_ratio': nodata_ratio})
        finally:
            for src in terrain_sources.values():
                src.close()
    return rows


## Section 4.2 — Batch Patching

Generating patches across all dates creates the training inventory used by every model variant. The saved patch manifest also gives us a reusable dataset summary for later notebooks and the final report.


In [ ]:
patch_rows = []
for vv_file in tqdm(sorted(PROCESSED_SAR.glob('S1_VV_*.tif')), desc='Generating patches'):
    date_token = vv_file.stem.split('_')[-1]
    vh_file = PROCESSED_SAR / f'S1_VH_{date_token}.tif'
    mask_file = PROCESSED_MASKS / f'mask_{date_token}.tif'
    rows = generate_patches(vv_file, vh_file, terrain_dict, mask_file, PROCESSED_PATCHES)
    patch_rows.extend(rows)
    print(f'{date_token}: {len(rows)} patches')
patch_manifest_df = pd.DataFrame(patch_rows)
patch_manifest_df.to_csv(PROCESSED_PATCHES / 'patch_manifest.csv', index=False)
display(patch_manifest_df.head())


## Section 4.3 — Train/Val/Test Split

A stratified split keeps flood-heavy and flood-light patches reasonably balanced across train, validation, and test subsets. That makes the ablation study comparisons more trustworthy.


In [ ]:
patch_manifest_df['flood_bin'] = pd.qcut(patch_manifest_df['flood_pixel_pct'], q=min(5, patch_manifest_df['flood_pixel_pct'].nunique()), duplicates='drop').astype(str)
train_df, temp_df = train_test_split(patch_manifest_df, test_size=0.30, random_state=SEED, stratify=patch_manifest_df['flood_bin'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['flood_bin'])
split_index = {'train': train_df['patch_path'].tolist(), 'val': val_df['patch_path'].tolist(), 'test': test_df['patch_path'].tolist()}
(PROCESSED_PATCHES / 'split_index.json').write_text(json.dumps(split_index, indent=2))
split_summary = pd.DataFrame([
    {'split': 'train', 'count': len(train_df), 'mean_flood_pct': train_df['flood_pixel_pct'].mean()},
    {'split': 'val', 'count': len(val_df), 'mean_flood_pct': val_df['flood_pixel_pct'].mean()},
    {'split': 'test', 'count': len(test_df), 'mean_flood_pct': test_df['flood_pixel_pct'].mean()},
])
display(split_summary)


## Section 4.4 — Augmentation Preview

The preview confirms that the same spatial transformation is applied to VV, VH, terrain, and mask together. That consistency is essential for segmentation accuracy.


In [ ]:
aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5)], additional_targets={'vh': 'image', 'terrain': 'image', 'mask': 'mask'})
sample = np.load(train_df.iloc[0]['patch_path'])
result = aug(image=sample['vv'][0], vh=sample['vh'][0], terrain=np.moveaxis(sample['terrain'], 0, -1), mask=sample['mask'][0])
vv_aug = result['image'] + np.random.normal(0.0, 0.05, size=result['image'].shape)
vh_aug = result['vh'] + np.random.normal(0.0, 0.05, size=result['vh'].shape)
terrain_aug = result['terrain']
fig, axes = plt.subplots(2, 3, figsize=(12, 8), dpi=300)
axes[0, 0].imshow(sample['vv'][0], cmap='gray'); axes[0, 0].set_title('VV original')
axes[0, 1].imshow(sample['vh'][0], cmap='gray'); axes[0, 1].set_title('VH original')
axes[0, 2].imshow(sample['terrain'][0], cmap='terrain'); axes[0, 2].set_title('Slope original')
axes[1, 0].imshow(vv_aug, cmap='gray'); axes[1, 0].set_title('VV augmented')
axes[1, 1].imshow(vh_aug, cmap='gray'); axes[1, 1].set_title('VH augmented')
axes[1, 2].imshow(terrain_aug[..., 0], cmap='terrain'); axes[1, 2].set_title('Slope augmented')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'augmentation_preview.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)


## Section 4.5 — Dataset Statistics

A final summary of patch counts and flood-percentage distributions gives a clean hand-off into the training notebook. It also creates reusable report metadata for the README and submission package.


In [ ]:
stats_df = pd.DataFrame([
    {'split': 'train', 'patches': len(train_df), 'min_flood_pct': train_df['flood_pixel_pct'].min(), 'max_flood_pct': train_df['flood_pixel_pct'].max(), 'mean_flood_pct': train_df['flood_pixel_pct'].mean()},
    {'split': 'val', 'patches': len(val_df), 'min_flood_pct': val_df['flood_pixel_pct'].min(), 'max_flood_pct': val_df['flood_pixel_pct'].max(), 'mean_flood_pct': val_df['flood_pixel_pct'].mean()},
    {'split': 'test', 'patches': len(test_df), 'min_flood_pct': test_df['flood_pixel_pct'].min(), 'max_flood_pct': test_df['flood_pixel_pct'].max(), 'mean_flood_pct': test_df['flood_pixel_pct'].mean()},
])
display(stats_df)
fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
ax.hist(train_df['flood_pixel_pct'], bins=20, alpha=0.7, label='Train', color='#264653')
ax.hist(val_df['flood_pixel_pct'], bins=20, alpha=0.7, label='Val', color='#2a9d8f')
ax.hist(test_df['flood_pixel_pct'], bins=20, alpha=0.7, label='Test', color='#e76f51')
ax.set_title('Flood percentage distribution across splits')
ax.set_xlabel('Flood pixel percentage')
ax.set_ylabel('Patch count')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'patch_split_histogram.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)
stats_df.to_csv(REPORT_DIR / 'patch_dataset_statistics.csv', index=False)


<!-- MULTI-HAZARD EXTENSION GENERATED -->
## Section 4.6 ? Multi-Hazard Patch Export
This extension converts the existing patch dataset into a **multi-target patch format**. Flood is copied into the first supervised channel, while erosion and landslide channels are initialized and reserved for later labels.


In [ ]:
# MULTI-HAZARD EXTENSION GENERATED
import pandas as pd
from analysis.multi_hazard_support import load_hazard_catalog, ordered_hazard_ids, convert_patch_dataset_to_multihazard

ROOT = Path(r'f:\MAPATHON\sylhet_flood_2024')
hazard_catalog = load_hazard_catalog(ROOT / 'config' / 'hazard_catalog.json')
hazard_order = ordered_hazard_ids(hazard_catalog)

source_patch_dir = ROOT / 'data' / 'processed' / 'patches_full256'
if not source_patch_dir.exists():
    source_patch_dir = ROOT / 'data' / 'processed' / 'patches'
target_patch_dir = ROOT / 'data' / 'processed' / 'patches_multihazard_ready'

if source_patch_dir.exists():
    multi_hazard_patch_manifest = convert_patch_dataset_to_multihazard(source_patch_dir, target_patch_dir, hazard_order)
    multi_hazard_patch_manifest.to_csv(ROOT / 'outputs' / 'report' / 'multi_hazard_patch_manifest.csv', index=False)
    multi_hazard_patch_manifest.head()
else:
    print('No source patch directory found. Expected patches_full256/ or patches/.')
